In [1]:
! pip install pandas
! pip install openpyxl

In [2]:
import pandas as pd
import json

In [3]:
file_path = '../data/xl/dados_metas_atualizados.xlsx'

In [4]:
df = pd.read_excel(file_path, engine='openpyxl', sheet_name='Temas, Planos e ODS')

In [5]:
df.head()

,Número da Meta,Destaque do Título,Descrição da Meta,Projeção,Indicador,Unnamed: 5,Secretaria,Eixo,Temas,Subprefeituras,ODS,Nº da Meta ODS,Planos Municipais
0,1,Entregar 8 novos parques,"Entregar 8 novos parques, para ampliar a ofert...","Com essa medida, a cidade chegará a 125 parque...",Número de parques entregues.,NaN,SVMA,Universo SP,Natureza e ambiente,Campo Limpo; Casa Verde/Cachoeirinha; Cidade A...,11; 13; 15,11.7; 15.1; 13.1,Plano Diretor Estratégico; Plano Municipal de ...
1,2,Requalificar ou ampliar 25 parques,"Requalificar ou ampliar 25 parques, com a melh...",Serão priorizadas as unidades com indicadores ...,Número de parques revitalizados.,NaN,SVMA,Universo SP,Natureza e ambiente,Butantã; Capão Redondo; Ermelino Matarazzo; It...,11; 13; 15,11.7; 15.1; 13.1,Plano Diretor Estratégico; Plano Municipal de ...
2,3,Realizar o projeto São Paulo +Verde,"Realizar o projeto São Paulo +Verde, que vai p...",A iniciativa envolve o plantio de árvores em p...,Número de árvores plantadas.,NaN,SVMA; SMSUB,Universo SP,Natureza e ambiente,NaN,2; 11; 13; 15,2.5; 11.7; 13.1; 15.1; 15.2,Plano Municipal de Arborização Urbana
3,4,"Alcançar 11,4% de tratamento de materiais reci...","Alcançar 11,4% de tratamento de materiais reci...",O aumento da capacidade de separação de resídu...,Percentual de resíduos recicláveis não destina...,NaN,SP REGULA; SMSUB; SMDET,Universo SP,Natureza e ambiente,NaN,12; 16,12.5; 16.6,Plano de Gestão Integrada de Resíduos Sólidos
4,5,Levar atividades de conscientização e educação...,Levar atividades de conscientização e educação...,Além de cursos e atividades formativas da Univ...,Número de atendimentos em ações de educação am...,NaN,SVMA,Universo SP,Natureza e ambiente,NaN,3; 9; 16,3.6; 9.1; 16.6,Plano Municipal de Educação Ambiental


In [6]:
renomear = {
    'Número da Meta' : 'numero',
    'Subprefeituras' : 'subprefeituras'
}

In [7]:
df = df.rename(columns=renomear)
df = df[renomear.values()].copy()

In [8]:
df.head()

,numero,subprefeituras
0,1,Campo Limpo; Casa Verde/Cachoeirinha; Cidade A...
1,2,Butantã; Capão Redondo; Ermelino Matarazzo; It...
2,3,NaN
3,4,NaN
4,5,NaN


In [9]:
df = df[df['subprefeituras'].notna()].reset_index(drop=True).copy()

In [10]:
replace_names = {'Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Butantã': 'BUTANTA',
 'São Mateus': 'SAO MATEUS',
 'Jaçanã/Tremembé': 'JACANA-TREMEMBE',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
 'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
 'Freguesia/Brasilândia': 'FREGUESIA-BRASILANDIA',
 'Pirituba/Jaraguá': 'PIRITUBA-JARAGUA',
 'São Miguel Paulista': 'SAO MIGUEL',
 'Aricanduva/Formosa/Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
 'Campo Limpo Itaquera': 'CAMPO LIMPO',
 'Sé' : "SE",
 'Capão Redondo' : 'CAMPO LIMPO'}

In [11]:
for atual, novo in replace_names.items():
    df['subprefeituras'] = df['subprefeituras'].str.replace(atual, novo)

Tenho que pegar a sigla a partir do nome da subprefeitura...

In [12]:
with open('../../cadastros_basicos/data/subprefeituras.json', 'r') as f:
    subprefeituras = json.load(f)

In [13]:
nomes_siglas_subs = {}

for sub_data in subprefeituras['features']:
    nome = sub_data['properties']['nm_subprefeitura']
    sigla = sub_data['properties']['sg_subprefeitura']
    nomes_siglas_subs[nome] = sigla

In [14]:
nomes_siglas_subs

{'CIDADE ADEMAR': 'AD',
 'MOOCA': 'MO',
 'IPIRANGA': 'IP',
 'LAPA': 'LA',
 'JABAQUARA': 'JA',
 'GUAIANASES': 'GU',
 'SANTO AMARO': 'SA',
 'SAPOPEMBA': 'SB',
 'SAO MIGUEL': 'MP',
 'JACANA-TREMEMBE': 'JT',
 'CASA VERDE-CACHOEIRINHA': 'CV',
 'CAMPO LIMPO': 'CL',
 'ITAIM PAULISTA': 'IT',
 'BUTANTA': 'BT',
 'SE': 'SE',
 'ITAQUERA': 'IQ',
 'SAO MATEUS': 'SM',
 'PERUS': 'PR',
 'CIDADE TIRADENTES': 'CT',
 'VILA MARIANA': 'VM',
 'VILA PRUDENTE': 'VP',
 'PINHEIROS': 'PI',
 'PARELHEIROS': 'PA',
 'FREGUESIA-BRASILANDIA': 'FO',
 'PENHA': 'PE',
 'ARICANDUVA-FORMOSA-CARRAO': 'AF',
 'SANTANA-TUCURUVI': 'ST',
 'VILA MARIA-VILA GUILHERME': 'MG',
 'M BOI MIRIM': 'MB',
 'PIRITUBA-JARAGUA': 'PJ',
 'ERMELINO MATARAZZO': 'EM',
 'CAPELA DO SOCORRO': 'CS'}

In [15]:
import difflib

def achar_nome_parecido(nome, nomes_siglas_subs, n=1, cutoff=0.6):
    nomes = list(nomes_siglas_subs.keys())
    similares = difflib.get_close_matches(nome.upper(), nomes, n=n, cutoff=cutoff)
    return similares

In [16]:
dados_parsed = {}
nao_encontrados = {}

for i, row in df.iterrows():
    numero = row['numero']
    subs = [s.strip() for s in row['subprefeituras'].split(';')]
    for s in subs:
        sub_upper = s.upper()
        if sub_upper not in nomes_siglas_subs and len(sub_upper)>1:
            print(f'Nome não encontrado: {s} (meta {numero})')
            nome_parecido = achar_nome_parecido(s, nomes_siglas_subs)
            if nome_parecido:
                print(f'Nome parecido encontrado: {nome_parecido[0]} (meta {numero})')
                nao_encontrados[s] = nome_parecido[0]
    siglas = [nomes_siglas_subs[s.upper()] for s in subs if s.upper() in nomes_siglas_subs and len(s)>1]
    dados_parsed[numero] = siglas

In [17]:
nao_encontrados

{}

In [18]:
dados_parsed

{1: ['CL', 'CV', 'AD', 'MO', 'SB'],
 2: ['BT', 'CL', 'EM', 'IQ', 'LA', 'SM', 'VM'],
 11: ['BT',
  'CT',
  'IP',
  'JA',
  'JT',
  'MB',
  'MO',
  'PE',
  'ST',
  'SM',
  'SE',
  'MG',
  'VM'],
 12: ['BT', 'CL', 'CV', 'AD', 'EM', 'FO', 'IQ', 'LA', 'MB', 'SB'],
 13: ['BT', 'CL', 'FO', 'IQ', 'PR', 'PJ', 'MP', 'VM', 'VP'],
 15: ['AF', 'IQ', 'MO', 'PE', 'SM'],
 16: ['AF', 'MO', 'PE', 'SE'],
 17: ['SE', 'MO'],
 18: ['AF', 'IQ', 'PE'],
 19: ['PE', 'EM'],
 20: ['CV', 'ST'],
 21: ['CL', 'MB'],
 22: ['CS', 'AD', 'SA'],
 23: ['AF', 'CV', 'CT', 'IP', 'IQ', 'LA', 'PE', 'PR', 'MP', 'SB', 'SE', 'VP'],
 24: ['IQ'],
 25: ['IT'],
 26: ['PR'],
 27: ['AD'],
 28: ['CS'],
 29: ['AD', 'CS'],
 30: ['MB'],
 31: ['MB'],
 32: ['PA'],
 33: ['SA', 'CS'],
 34: ['CS'],
 35: ['CS'],
 36: ['SA'],
 37: ['JA'],
 38: ['PI'],
 39: ['PI'],
 40: ['PI'],
 41: ['VM'],
 42: ['MP'],
 43: ['LA', 'PJ'],
 44: ['SE', 'MO'],
 48: ['MG'],
 52: ['CT', 'FO', 'IP', 'LA', 'SE'],
 53: ['SE'],
 54: ['SE'],
 63: ['BT',
  'CL',
  'CS',
  'AD

In [19]:
with open('../data/metas_subs.json', 'w') as f:
    json.dump(dados_parsed, f, ensure_ascii=False, indent=4)